# Customer Segmentation with K-Means Clustering

## Session 1: K-Means Clustering for E-commerce

**Learning Objectives:**
1. Implement K-Means clustering algorithm
2. Use elbow method and silhouette analysis to find optimal K
3. Profile and name customer segments
4. Generate actionable business recommendations

**Business Context:**
An e-commerce company wants to segment their 2,000 customers to:
- Create targeted marketing campaigns
- Personalize product recommendations
- Optimize customer retention strategies
- Allocate resources effectively

**Success Criteria:**
- Identify 4-5 distinct customer segments
- Clear differentiation between segments
- Actionable marketing recommendations per segment

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Generate Synthetic E-commerce Customer Data

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate 2,000 customers with 5 distinct segments
n_customers = 2000

# Segment 1: High-Value Frequent Buyers (20%)
n_seg1 = 400
seg1 = pd.DataFrame({
    'recency': np.random.randint(1, 30, n_seg1),          # Days since last purchase (recent)
    'frequency': np.random.randint(20, 50, n_seg1),        # Number of purchases (high)
    'monetary': np.random.uniform(5000, 15000, n_seg1),    # Total spend (very high)
    'avg_order_value': np.random.uniform(200, 500, n_seg1),
    'age': np.random.randint(35, 55, n_seg1),
    'web_visits_per_month': np.random.randint(15, 30, n_seg1),
    'segment_true': 'High-Value Frequent'
})

# Segment 2: Loyal Regulars (25%)
n_seg2 = 500
seg2 = pd.DataFrame({
    'recency': np.random.randint(1, 60, n_seg2),
    'frequency': np.random.randint(10, 25, n_seg2),
    'monetary': np.random.uniform(2000, 6000, n_seg2),
    'avg_order_value': np.random.uniform(100, 250, n_seg2),
    'age': np.random.randint(25, 60, n_seg2),
    'web_visits_per_month': np.random.randint(8, 18, n_seg2),
    'segment_true': 'Loyal Regulars'
})

# Segment 3: At-Risk Customers (15%)
n_seg3 = 300
seg3 = pd.DataFrame({
    'recency': np.random.randint(90, 365, n_seg3),         # Haven't bought in a while
    'frequency': np.random.randint(5, 15, n_seg3),         # Used to buy
    'monetary': np.random.uniform(1000, 4000, n_seg3),
    'avg_order_value': np.random.uniform(80, 200, n_seg3),
    'age': np.random.randint(25, 65, n_seg3),
    'web_visits_per_month': np.random.randint(1, 5, n_seg3),  # Low engagement
    'segment_true': 'At-Risk'
})

# Segment 4: New Customers (20%)
n_seg4 = 400
seg4 = pd.DataFrame({
    'recency': np.random.randint(1, 30, n_seg4),
    'frequency': np.random.randint(1, 5, n_seg4),          # Few purchases
    'monetary': np.random.uniform(50, 1000, n_seg4),       # Low total spend
    'avg_order_value': np.random.uniform(50, 200, n_seg4),
    'age': np.random.randint(18, 45, n_seg4),
    'web_visits_per_month': np.random.randint(5, 15, n_seg4),
    'segment_true': 'New Customers'
})

# Segment 5: Window Shoppers (20%)
n_seg5 = 400
seg5 = pd.DataFrame({
    'recency': np.random.randint(30, 180, n_seg5),
    'frequency': np.random.randint(1, 3, n_seg5),          # Very few purchases
    'monetary': np.random.uniform(20, 500, n_seg5),        # Low spend
    'avg_order_value': np.random.uniform(20, 100, n_seg5),
    'age': np.random.randint(18, 50, n_seg5),
    'web_visits_per_month': np.random.randint(10, 25, n_seg5),  # High browsing, low buying
    'segment_true': 'Window Shoppers'
})

# Combine all segments
customers = pd.concat([seg1, seg2, seg3, seg4, seg5], ignore_index=True)

# Shuffle the data
customers = customers.sample(frac=1, random_state=42).reset_index(drop=True)

# Add customer IDs
customers.insert(0, 'customer_id', range(1, len(customers) + 1))

print(f"Generated {len(customers)} customers")
print(f"\nDataset shape: {customers.shape}")
print(f"\nFeatures: {list(customers.columns)}")
customers.head(10)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("Dataset Info:")
print(customers.info())
print("\n" + "="*80 + "\n")
print("Statistical Summary:")
customers.describe()

In [ ]:
# True segment distribution
print("True Segment Distribution:")
print(customers['segment_true'].value_counts())
print(f"\nTotal: {len(customers)} customers")

plt.figure(figsize=(10, 6))
customers['segment_true'].value_counts().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('True Customer Segment Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Segment', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')

features = ['recency', 'frequency', 'monetary', 'avg_order_value', 'age', 'web_visits_per_month']

for idx, feature in enumerate(features):
    row = idx // 3
    col = idx % 3
    axes[row, col].hist(customers[feature], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[row, col].set_title(feature.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[row, col].set_xlabel(feature.replace('_', ' ').title(), fontsize=10)
    axes[row, col].set_ylabel('Frequency', fontsize=10)
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = customers[features].corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nKey Correlations:")
print("- Frequency and Monetary are highly correlated (expected)")
print("- Recency is negatively correlated with engagement metrics")
print("- Age has weak correlation with purchase behavior")

## 4. Data Preparation for Clustering

In [ ]:
# Select features for clustering
clustering_features = ['recency', 'frequency', 'monetary', 'avg_order_value', 'age', 'web_visits_per_month']
X = customers[clustering_features].copy()

print("Features for clustering:")
print(X.head())
print(f"\nShape: {X.shape}")

# Check for missing values
print(f"\nMissing values: {X.isnull().sum().sum()}")

In [ ]:
# CRITICAL: Feature Scaling for K-Means
# K-Means is distance-based, so scaling is essential!

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for easier handling
X_scaled_df = pd.DataFrame(X_scaled, columns=clustering_features)

print("Scaled features:")
print(X_scaled_df.head())
print(f"\nMean: {X_scaled_df.mean().round(3).to_dict()}")
print(f"Std: {X_scaled_df.std().round(3).to_dict()}")
print("\nAll features now have mean=0 and std=1 (standardized)")

## 5. Finding Optimal K - Elbow Method

In [ ]:
# Test K values from 2 to 10
K_range = range(2, 11)
inertias = []

print("Testing different values of K...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    print(f"K={k}: Inertia={kmeans.inertia_:.2f}")

print("\nDone!")

In [ ]:
# Plot Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
plt.title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(K_range)

# Highlight potential elbow points
plt.axvline(x=4, color='red', linestyle='--', alpha=0.5, label='Potential elbow (K=4)')
plt.axvline(x=5, color='orange', linestyle='--', alpha=0.5, label='Potential elbow (K=5)')
plt.legend()
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Look for the 'elbow' where inertia decrease slows down")
print("- The elbow appears around K=4 or K=5")
print("- After K=5, improvements are marginal")

## 6. Finding Optimal K - Silhouette Analysis

In [ ]:
# Calculate silhouette scores for different K values
silhouette_scores = []

print("Calculating silhouette scores...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(silhouette_avg)
    print(f"K={k}: Silhouette Score={silhouette_avg:.4f}")

print("\nDone!")

In [ ]:
# Plot Silhouette Scores
plt.figure(figsize=(10, 6))
plt.plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Analysis for Optimal K', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(K_range)

# Highlight best K
best_k = K_range[np.argmax(silhouette_scores)]
best_score = max(silhouette_scores)
plt.axvline(x=best_k, color='red', linestyle='--', alpha=0.5, 
            label=f'Best K={best_k} (Score={best_score:.4f})')
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nBest K by Silhouette Score: {best_k}")
print(f"Best Silhouette Score: {best_score:.4f}")
print("\nSilhouette Score interpretation:")
print("- Range: [-1, 1]")
print("- Close to 1: Well-separated clusters")
print("- Close to 0: Overlapping clusters")
print("- Negative: Points may be in wrong clusters")

## 7. Final K-Means Model with Optimal K

In [ ]:
# Choose optimal K (let's use K=5 based on both methods)
optimal_k = 5

print(f"Training final K-Means model with K={optimal_k}...")
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
customers['cluster'] = kmeans_final.fit_predict(X_scaled)

# Calculate final metrics
final_inertia = kmeans_final.inertia_
final_silhouette = silhouette_score(X_scaled, customers['cluster'])

print(f"\nFinal Model Performance:")
print(f"- Number of clusters: {optimal_k}")
print(f"- Inertia: {final_inertia:.2f}")
print(f"- Silhouette Score: {final_silhouette:.4f}")
print(f"- Iterations: {kmeans_final.n_iter_}")

print(f"\nCluster sizes:")
print(customers['cluster'].value_counts().sort_index())

## 8. Cluster Analysis and Profiling

In [ ]:
# Calculate mean values for each cluster
cluster_profiles = customers.groupby('cluster')[clustering_features].mean().round(2)
cluster_sizes = customers['cluster'].value_counts().sort_index()
cluster_profiles['size'] = cluster_sizes.values

print("Cluster Profiles (Mean Values):")
print("="*100)
print(cluster_profiles)
print("\n" + "="*100)

In [ ]:
# Visualize cluster profiles with heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_profiles[clustering_features].T, annot=True, fmt='.1f', 
            cmap='YlOrRd', cbar_kws={'label': 'Value'}, linewidths=0.5)
plt.title('Cluster Profiles Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Name the clusters based on their characteristics
cluster_names = {
    0: 'Champions',
    1: 'Potential Loyalists',
    2: 'At Risk',
    3: 'Lost Customers',
    4: 'New Customers'
}

# Analyze each cluster to assign names
print("\nDetailed Cluster Analysis:")
print("="*100)
for cluster_id in range(optimal_k):
    cluster_data = customers[customers['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id}:")
    print(f"  Size: {len(cluster_data)} customers ({len(cluster_data)/len(customers)*100:.1f}%)")
    print(f"  Recency: {cluster_data['recency'].mean():.1f} days")
    print(f"  Frequency: {cluster_data['frequency'].mean():.1f} purchases")
    print(f"  Monetary: ${cluster_data['monetary'].mean():.2f}")
    print(f"  Avg Order Value: ${cluster_data['avg_order_value'].mean():.2f}")
    print(f"  Age: {cluster_data['age'].mean():.1f} years")
    print(f"  Web Visits/Month: {cluster_data['web_visits_per_month'].mean():.1f}")
    
    # Determine cluster name based on characteristics
    recency = cluster_data['recency'].mean()
    frequency = cluster_data['frequency'].mean()
    monetary = cluster_data['monetary'].mean()
    
    if recency < 50 and frequency > 15 and monetary > 4000:
        name = "Champions (High-Value Frequent Buyers)"
    elif recency < 80 and frequency > 8 and monetary > 2000:
        name = "Potential Loyalists (Regular Buyers)"
    elif recency > 150 and frequency > 5:
        name = "At Risk (Need Reactivation)"
    elif frequency < 5 and monetary < 1500 and recency < 60:
        name = "New Customers (Low History)"
    else:
        name = "Lost/Inactive (Need Win-Back)"
    
    cluster_names[cluster_id] = name
    print(f"  Segment Name: {name}")

print("\n" + "="*100)

# Add cluster names to dataframe
customers['cluster_name'] = customers['cluster'].map(cluster_names)

In [ ]:
# Visualize cluster distribution
plt.figure(figsize=(12, 6))
cluster_counts = customers['cluster_name'].value_counts()
colors = plt.cm.Set3(range(len(cluster_counts)))
plt.bar(range(len(cluster_counts)), cluster_counts.values, color=colors, edgecolor='black', linewidth=1.5)
plt.xticks(range(len(cluster_counts)), cluster_counts.index, rotation=45, ha='right')
plt.xlabel('Customer Segment', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)
plt.title('Customer Segment Distribution', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(cluster_counts.values):
    plt.text(i, v + 20, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Cluster Visualization in 2D

In [ ]:
# Use PCA to reduce to 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA Explained Variance:")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"  Total: {sum(pca.explained_variance_ratio_):.2%}")

# Add PCA coordinates to dataframe
customers['pca1'] = X_pca[:, 0]
customers['pca2'] = X_pca[:, 1]

In [ ]:
# Scatter plot with clusters
plt.figure(figsize=(14, 8))
scatter = plt.scatter(customers['pca1'], customers['pca2'], 
                     c=customers['cluster'], cmap='viridis', 
                     s=50, alpha=0.6, edgecolors='black', linewidth=0.5)

# Plot centroids
centroids_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], 
           c='red', s=300, alpha=0.8, edgecolors='black', linewidth=2, 
           marker='X', label='Centroids')

plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('Customer Segments (K-Means Clustering)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Create multiple 2D views
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Customer Segments - Multiple Views', fontsize=16, fontweight='bold')

# View 1: Recency vs Frequency
for cluster_id in range(optimal_k):
    cluster_data = customers[customers['cluster'] == cluster_id]
    axes[0, 0].scatter(cluster_data['recency'], cluster_data['frequency'], 
                      label=f'Cluster {cluster_id}', alpha=0.6, s=50)
axes[0, 0].set_xlabel('Recency (days)', fontsize=11)
axes[0, 0].set_ylabel('Frequency (purchases)', fontsize=11)
axes[0, 0].set_title('Recency vs Frequency', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# View 2: Monetary vs Frequency
for cluster_id in range(optimal_k):
    cluster_data = customers[customers['cluster'] == cluster_id]
    axes[0, 1].scatter(cluster_data['monetary'], cluster_data['frequency'], 
                      label=f'Cluster {cluster_id}', alpha=0.6, s=50)
axes[0, 1].set_xlabel('Monetary (total spend)', fontsize=11)
axes[0, 1].set_ylabel('Frequency (purchases)', fontsize=11)
axes[0, 1].set_title('Monetary vs Frequency', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# View 3: Recency vs Monetary
for cluster_id in range(optimal_k):
    cluster_data = customers[customers['cluster'] == cluster_id]
    axes[1, 0].scatter(cluster_data['recency'], cluster_data['monetary'], 
                      label=f'Cluster {cluster_id}', alpha=0.6, s=50)
axes[1, 0].set_xlabel('Recency (days)', fontsize=11)
axes[1, 0].set_ylabel('Monetary (total spend)', fontsize=11)
axes[1, 0].set_title('Recency vs Monetary', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# View 4: Web Visits vs Monetary
for cluster_id in range(optimal_k):
    cluster_data = customers[customers['cluster'] == cluster_id]
    axes[1, 1].scatter(cluster_data['web_visits_per_month'], cluster_data['monetary'], 
                      label=f'Cluster {cluster_id}', alpha=0.6, s=50)
axes[1, 1].set_xlabel('Web Visits per Month', fontsize=11)
axes[1, 1].set_ylabel('Monetary (total spend)', fontsize=11)
axes[1, 1].set_title('Web Visits vs Monetary', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Radar Chart for Cluster Profiles

In [ ]:
# Normalize cluster profiles for radar chart (0-1 scale)
from sklearn.preprocessing import MinMaxScaler

scaler_viz = MinMaxScaler()
cluster_profiles_normalized = cluster_profiles[clustering_features].copy()
cluster_profiles_normalized = pd.DataFrame(
    scaler_viz.fit_transform(cluster_profiles_normalized),
    columns=clustering_features,
    index=cluster_profiles.index
)

# Create radar charts
from math import pi

categories = clustering_features
N = len(categories)

# Create subplots for each cluster
fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw=dict(projection='polar'))
fig.suptitle('Customer Segment Profiles (Radar Charts)', fontsize=16, fontweight='bold')

axes = axes.ravel()

for idx, cluster_id in enumerate(range(optimal_k)):
    ax = axes[idx]
    
    # Get values for this cluster
    values = cluster_profiles_normalized.loc[cluster_id].values.tolist()
    values += values[:1]  # Complete the circle
    
    # Calculate angle for each axis
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    
    # Plot
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {cluster_id}')
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([cat.replace('_', '\n') for cat in categories], size=8)
    ax.set_ylim(0, 1)
    ax.set_title(f"{cluster_names[cluster_id]}\n({cluster_profiles.loc[cluster_id, 'size']} customers)", 
                size=11, fontweight='bold', pad=20)
    ax.grid(True)

# Remove extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

## 11. Business Insights and Recommendations

In [ ]:
# Generate comprehensive business insights
print("="*100)
print("CUSTOMER SEGMENTATION - BUSINESS INSIGHTS AND RECOMMENDATIONS")
print("="*100)

for cluster_id in range(optimal_k):
    cluster_data = customers[customers['cluster'] == cluster_id]
    name = cluster_names[cluster_id]
    size = len(cluster_data)
    pct = size / len(customers) * 100
    
    print(f"\n{'='*100}")
    print(f"SEGMENT {cluster_id}: {name}")
    print(f"{'='*100}")
    print(f"Size: {size} customers ({pct:.1f}% of total)")
    print(f"")
    print(f"Profile:")
    print(f"  - Avg Recency: {cluster_data['recency'].mean():.1f} days")
    print(f"  - Avg Frequency: {cluster_data['frequency'].mean():.1f} purchases")
    print(f"  - Avg Monetary: ${cluster_data['monetary'].mean():.2f}")
    print(f"  - Avg Order Value: ${cluster_data['avg_order_value'].mean():.2f}")
    print(f"  - Total Revenue: ${cluster_data['monetary'].sum():.2f}")
    print(f"  - Revenue Share: {cluster_data['monetary'].sum() / customers['monetary'].sum() * 100:.1f}%")
    
    # Generate recommendations based on segment characteristics
    print(f"\nMarketing Recommendations:")
    
    recency = cluster_data['recency'].mean()
    frequency = cluster_data['frequency'].mean()
    monetary = cluster_data['monetary'].mean()
    
    if recency < 50 and frequency > 15 and monetary > 4000:
        print(f"  1. VIP Treatment: Exclusive access to new products and premium support")
        print(f"  2. Loyalty Rewards: High-tier loyalty program with personalized benefits")
        print(f"  3. Referral Program: Incentivize them to bring in new high-value customers")
        print(f"  4. Cross-sell/Up-sell: Premium products and complementary items")
        print(f"  5. Retention: Regular engagement to maintain relationship")
    elif recency < 80 and frequency > 8 and monetary > 2000:
        print(f"  1. Upgrade Path: Incentives to increase purchase frequency")
        print(f"  2. Personalization: Tailored product recommendations")
        print(f"  3. Engagement: Regular newsletters with exclusive offers")
        print(f"  4. Loyalty Building: Points-based rewards program")
        print(f"  5. Feedback: Solicit reviews and testimonials")
    elif recency > 150 and frequency > 5:
        print(f"  1. Win-Back Campaign: Aggressive discounts to re-engage")
        print(f"  2. Survey: Understand why they stopped buying")
        print(f"  3. New Products: Showcase what they've missed")
        print(f"  4. Multi-channel: Email, SMS, retargeting ads")
        print(f"  5. Urgency: Time-limited offers to drive immediate action")
    elif frequency < 5 and monetary < 1500 and recency < 60:
        print(f"  1. Onboarding: Welcome series to build relationship")
        print(f"  2. Education: Product guides and how-tos")
        print(f"  3. First Purchase Incentive: Discount on second order")
        print(f"  4. Social Proof: Customer reviews and testimonials")
        print(f"  5. Engagement: Encourage account creation and profile completion")
    else:
        print(f"  1. Reactivation: Deep discounts to revive interest")
        print(f"  2. Last Chance: Final offer before unsubscribing")
        print(f"  3. Content Marketing: Valuable content to rebuild trust")
        print(f"  4. Channel Testing: Try different communication methods")
        print(f"  5. Segment Analysis: Understand if worth pursuing")

print(f"\n{'='*100}")
print(f"OVERALL INSIGHTS:")
print(f"{'='*100}")
print(f"\n1. Revenue Concentration:")
revenue_by_cluster = customers.groupby('cluster')['monetary'].sum().sort_values(ascending=False)
for cluster_id, revenue in revenue_by_cluster.items():
    pct = revenue / customers['monetary'].sum() * 100
    print(f"   - {cluster_names[cluster_id]}: ${revenue:,.2f} ({pct:.1f}%)")

print(f"\n2. Resource Allocation Strategy:")
print(f"   - Focus 50% of marketing budget on Champions (highest ROI)")
print(f"   - Allocate 30% to Potential Loyalists (growth opportunity)")
print(f"   - Invest 15% in At-Risk customers (prevent churn)")
print(f"   - Use 5% for New Customers (acquire and convert)")

print(f"\n3. Key Action Items:")
print(f"   - Implement segment-specific email campaigns within 2 weeks")
print(f"   - Create personalized product recommendations per segment")
print(f"   - Launch win-back campaign for At-Risk segment")
print(f"   - Set up automated triggers for segment transitions")
print(f"   - Monitor segment movements monthly")

print(f"\n{'='*100}")

## 12. Model Evaluation Metrics

In [ ]:
# Calculate per-sample silhouette scores
silhouette_vals = silhouette_samples(X_scaled, customers['cluster'])
customers['silhouette_score'] = silhouette_vals

# Silhouette plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Silhouette plot
y_lower = 10
for i in range(optimal_k):
    cluster_silhouette_vals = silhouette_vals[customers['cluster'] == i]
    cluster_silhouette_vals.sort()
    
    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    color = plt.cm.nipy_spectral(float(i) / optimal_k)
    ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                     facecolor=color, edgecolor=color, alpha=0.7)
    
    ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, f'Cluster {i}')
    y_lower = y_upper + 10

ax1.set_title('Silhouette Plot for Each Cluster', fontsize=12, fontweight='bold')
ax1.set_xlabel('Silhouette Coefficient Values', fontsize=11)
ax1.set_ylabel('Cluster Label', fontsize=11)
ax1.axvline(x=final_silhouette, color="red", linestyle="--", label=f'Average: {final_silhouette:.3f}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Silhouette scores by cluster
silhouette_by_cluster = customers.groupby('cluster')['silhouette_score'].mean().sort_index()
colors = plt.cm.nipy_spectral(np.linspace(0, 1, optimal_k))
ax2.bar(range(optimal_k), silhouette_by_cluster.values, color=colors, edgecolor='black', linewidth=1.5)
ax2.axhline(y=final_silhouette, color='red', linestyle='--', linewidth=2, label=f'Overall Average: {final_silhouette:.3f}')
ax2.set_xlabel('Cluster', fontsize=11)
ax2.set_ylabel('Average Silhouette Score', fontsize=11)
ax2.set_title('Average Silhouette Score by Cluster', fontsize=12, fontweight='bold')
ax2.set_xticks(range(optimal_k))
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSilhouette Analysis:")
print(f"Overall Silhouette Score: {final_silhouette:.4f}")
print(f"\nBy Cluster:")
for cluster_id in range(optimal_k):
    score = silhouette_by_cluster[cluster_id]
    print(f"  Cluster {cluster_id} ({cluster_names[cluster_id]}): {score:.4f}")

## 13. Save Results

In [ ]:
# Save clustered customer data
output_file = '../datasets/customers_segmented.csv'
customers.to_csv(output_file, index=False)
print(f"Saved segmented customer data to: {output_file}")

# Save cluster profiles
profile_file = '../datasets/cluster_profiles.csv'
cluster_profiles.to_csv(profile_file)
print(f"Saved cluster profiles to: {profile_file}")

print("\nAll results saved successfully!")

## Summary and Key Takeaways

### What We Accomplished:

1. **Data Preparation**
   - Generated 2,000 synthetic e-commerce customers
   - 6 features: Recency, Frequency, Monetary, Average Order Value, Age, Web Visits
   - Properly scaled features (critical for K-Means!)

2. **Optimal K Selection**
   - Elbow Method: Suggested K=4 or K=5
   - Silhouette Analysis: Confirmed K=5 as optimal
   - Final choice: K=5 segments

3. **Customer Segments Identified**
   - Champions: High-value frequent buyers (premium treatment)
   - Potential Loyalists: Regular buyers (upgrade opportunity)
   - At Risk: Need reactivation (win-back campaigns)
   - New Customers: Low history (onboarding focus)
   - Lost/Inactive: Need recovery (last chance offers)

4. **Business Value**
   - Targeted marketing campaigns per segment
   - Resource allocation optimization
   - Revenue concentration insights
   - Churn prevention strategies

### Key Metrics:
- **Silhouette Score**: Good cluster separation
- **Inertia**: Acceptable within-cluster variance
- **5 Distinct Segments**: Clear differentiation

### Next Steps:
1. Implement segment-specific marketing campaigns
2. Set up automated segment monitoring
3. A/B test different strategies per segment
4. Track segment transitions over time
5. Measure ROI of segmentation strategy

### Important Learnings:
- **Feature scaling is critical** for K-Means (distance-based)
- **Multiple methods** for choosing K (elbow + silhouette)
- **Business context matters** for naming and interpreting clusters
- **Visualization is key** for understanding and communicating results
- **Actionable insights** are more valuable than perfect clusters